<a href="https://colab.research.google.com/github/yeaye-hufs/DM_Cancer-vs-Normal-Classification-ML-Model/blob/main/DM_Project_1_data_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. GDC Client 리눅스 버전 다운로드 및 설치
!wget https://gdc.cancer.gov/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
!unzip gdc-client_v1.6.1_Ubuntu_x64.zip
!chmod +x gdc-client

# 2. 다운로드할 폴더 생성
import os
if not os.path.exists('tcga_data'):
    os.makedirs('tcga_data')

# 3. gdc-client 실행 (manifest 파일을 읽어 tcga_data 폴더에 저장)
# !./gdc-client download -m [매니페스트파일명] -d [저장폴더]
!./gdc-client download -m gdc_manifest.txt -d tcga_data

--2026-06-23 04:47:58--  https://gdc.cancer.gov/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
Resolving gdc.cancer.gov (gdc.cancer.gov)... 98.87.60.70, 54.162.107.110
Connecting to gdc.cancer.gov (gdc.cancer.gov)|98.87.60.70|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://gdc.cancer.gov:443/system/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip [following]
--2026-06-23 04:47:58--  https://gdc.cancer.gov/system/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
Reusing existing connection to gdc.cancer.gov:443.
HTTP request sent, awaiting response... 200 OK
Length: 23940006 (23M) [application/zip]
Saving to: ‘gdc-client_v1.6.1_Ubuntu_x64.zip’

gdc-client_v1.6.1_U 100%[===================>]  22.83M  28.3MB/s    in 0.8s    

2026-06-23 04:47:59 (28.3 MB/s) - ‘gdc-client_v1.6.1_Ubuntu_x64.zip’ saved [23940006/23940006]

Archive:  gdc-client_v1.6.1_Ubuntu_x64.zip
 extracting: gdc-client              
100% [##########################

In [ ]:
import os
import pandas as pd

data_dir = "tcga_data"
dfs = []

# 파일 목록 순회
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith(".tsv"):  # TSV 파일만 타겟
            file_path = os.path.join(root, file)

            # UUID(폴더명)를 샘플 이름으로 임시 사용
            sample_name = root.split('/')[-1]

            # skiprows=6: 주석 제거 / usecols: 유전자ID와 unstranded 발현량만 선택
            # 보통 0번 컬럼이 gene_id, 3번 컬럼이 unstranded 발현량입니다.
            try:
                df = pd.read_csv(file_path, sep='\t', skiprows=6, header=None, usecols=[0, 3])
                df.columns = ['gene', sample_name]
                df = df.set_index('gene')
                dfs.append(df)
            except:
                print(f"파일 읽기 실패: {file}")

# 한 번에 옆으로 이어 붙이기 (Merge보다 훨씬 빠름)
expr = pd.concat(dfs, axis=1)

# 불필요한 통계 행 제거 (N_unmapped 등)
expr = expr[~expr.index.str.startswith('_')]

# Gene ID 버전 정리 (ENSG000001.14 -> ENSG000001)
expr.index = expr.index.str.split('.').str[0]

# 중복 Gene 합계 처리
expr = expr.groupby(expr.index).sum()

print(f"최종 데이터 크기: {expr.shape}")
expr.head()

최종 데이터 크기: (60616, 84)


,9726aa3d-8a66-4568-92e5-c092c05a9a5d,1c5de7db-7e05-4289-ace6-b59b6e2392e7,94861d8a-3b40-4334-9fb5-c9df71a1e1da,56e9a740-ee72-4c10-a90d-921034141b42,2c81c69f-e08a-4586-b3f9-33d4af417d1f,3ce6800e-9df0-45a7-9e18-7e141911e4e9,3362c701-3af3-4947-983a-6039b67d1526,6c20a899-c806-46d0-b744-a26e53a9cb45,a5407526-9c45-427b-abcc-65d6e1f87158,e4bb1f41-bf42-43f3-aba5-98f3a02a7e03,...,201c0fbc-124c-4ef0-b767-393443a47892,a777fb14-4498-4f32-bccb-1e8521567302,a135a2b4-bc6a-4dd6-baa9-f08ab2d4971a,ba8aebd8-07de-46c0-a28f-b2408c36f07f,e6543906-a4e3-447c-8040-310979304204,fc732523-d387-4a1d-ab81-34e44e869bac,b5df8e57-5822-4192-a4a4-398d11e9d968,77819b24-ea90-427b-8a99-c9a90417c903,5fa5664e-daf5-4082-8d39-703a5d650e43,aab3583f-37ca-4f2f-8428-aa8eda2c2274
gene,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,4238,1157,4820,1745,1213,513,1329,2794,5010,6341,...,4199,2982,3136,4236,2255,401,3018,6039,2121,835
ENSG00000000005,0,2,3,2,0,0,1,0,917,31,...,5,0,0,1321,0,73,0,1,2,0
ENSG00000000419,2382,1235,1405,1381,1359,585,1259,1659,640,2299,...,2074,657,1460,1799,3896,154,1159,2198,1609,572
ENSG00000000457,1372,805,459,574,805,304,274,905,522,637,...,1952,341,950,1301,2317,221,444,1767,1181,274
ENSG00000000460,363,208,398,162,667,77,190,204,275,289,...,666,179,531,376,1258,102,323,689,580,93


In [ ]:
# 1. 통합된 표를 CSV 파일로 저장 (index=True는 유전자 ID를 포함한다는 뜻)
expr.to_csv('tcga_lung_cancer_counts.csv', index=True)

print("파일 생성 완료! 왼쪽 폴더 창에서 확인하세요.")

파일 생성 완료! 왼쪽 폴더 창에서 확인하세요.
